In [1]:
import pandas as pd

file_path = 'data/archive/accepted_2007_to_2018Q4.csv'

# Read only the first 5 lines
df_head = pd.read_csv(file_path, nrows=5, low_memory=False)
print("Total columns:", df_head.shape[1])
print("All column names are as follows：")
print(df_head.columns.tolist())


Total columns: 151
All column names are as follows：
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_jo

In [6]:
import pandas as pd
import os

# Set file paths
input_path = 'data/archive/accepted_2007_to_2018Q4.csv'  # Raw Lending Club Data
output_dir = 'preprocess/raw_data_analysis'
os.makedirs(output_dir, exist_ok=True)

# Sample and read first 10,000 rows for field exploration
sample_df = pd.read_csv(input_path, nrows=10000, low_memory=False)

# Build a field analysis summary
summary = pd.DataFrame({
    'field_name': sample_df.columns,
    'data_type': sample_df.dtypes.astype(str),
    'missing_rate': sample_df.isnull().mean().round(4),
    'num_unique_values': sample_df.nunique(),
    'sample_value': sample_df.iloc[0].astype(str).values
})

# Save summary as CSV
output_file = os.path.join(output_dir, 'field_structure_analysis.csv')
summary.to_csv(output_file, index=False, encoding='utf-8-sig')

print("Field structure analysis saved to:", output_file)


Field structure analysis saved to: preprocess/raw_data_analysis\field_structure_analysis.csv


In [7]:
import pandas as pd
from collections import Counter

file_path = 'data/archive/accepted_2007_to_2018Q4.csv'

# Initialize counter
loan_status_counter = Counter()

# Read loan_status in chunks
chunk_size = 500_000
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, usecols=['loan_status'], low_memory=False)):
    print(f"Analyzing chunk {i + 1}...")
    loan_status_counter.update(chunk['loan_status'].dropna())

# Display top 30 most common loan_status values
print("\n[Top 30 Most Common Values in 'loan_status']:")
for status, count in loan_status_counter.most_common(30):
    print(f"{status:<60} : {count:,}")


Analyzing chunk 1...
Analyzing chunk 2...
Analyzing chunk 3...
Analyzing chunk 4...
Analyzing chunk 5...

[Top 30 Most Common Values in 'loan_status']:
Fully Paid                                                   : 1,076,751
Current                                                      : 878,317
Charged Off                                                  : 268,559
Late (31-120 days)                                           : 21,467
In Grace Period                                              : 8,436
Late (16-30 days)                                            : 4,349
Does not meet the credit policy. Status:Fully Paid           : 1,988
Does not meet the credit policy. Status:Charged Off          : 761
Default                                                      : 40


In [11]:
import pandas as pd
import os

file_path = 'data/archive/accepted_2007_to_2018Q4.csv'

# Set output paths
X_file = 'preprocess/data_features/model_features/original/X_features.csv'
X_ext_file = 'preprocess/data_features/model_features/original/X_features_with_controversial.csv'
proxy_file = 'preprocess/data_features/proxy_features/proxy_features.csv'
y_file = 'preprocess/data_features/binary_target_labels/target.csv'

# Create directories if they don't exist
for path in [X_file, X_ext_file, proxy_file, y_file]:
    os.makedirs(os.path.dirname(path), exist_ok=True)

# Feature definitions
model_features = [
    'loan_amnt', 'funded_amnt', 'installment', 'annual_inc', 'emp_length',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec',
    'revol_util', 'total_acc', 'dti', 'purpose'
]
controversial_features = ['int_rate', 'verification_status']
proxy_features = ['zip_code', 'addr_state', 'home_ownership', 'application_type']
target = ['loan_status']

chunk_size = 500_000
all_columns = model_features + controversial_features + proxy_features + target
first_chunk = True

# Label mapping function (conservative strategy)
def map_loan_status_conservatively(status: str) -> int:
    status = str(status).strip().lower()
    if status in ['charged off', 'default', 'does not meet the credit policy. status:charged off']:
        return 1
    elif status in ['fully paid', 'does not meet the credit policy. status:fully paid']:
        return 0
    else:
        raise ValueError(f"Undefined state: {status}")

# Process in chunks
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, usecols=all_columns, low_memory=False)):
    print(f"Analyzing chunk {i + 1}...")

    valid_statuses = [
        'Charged Off', 'Default',
        'Does not meet the credit policy. Status: Charged Off',
        'Fully Paid', 'Does not meet the credit policy. Status: Fully Paid'
    ]
    chunk = chunk[chunk['loan_status'].isin(valid_statuses)].copy()
    chunk['loan_label'] = chunk['loan_status'].apply(map_loan_status_conservatively)

    X_chunk = chunk[model_features].copy()
    X_ext_chunk = chunk[model_features + controversial_features].copy()
    proxy_chunk = chunk[proxy_features].copy()
    y_chunk = chunk[['loan_label']].copy()

    for df in [X_chunk, X_ext_chunk]:
        if 'revol_util' in df.columns:
            df['revol_util'] = df['revol_util'].replace('%', '', regex=True)
            df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')

    mode = 'w' if first_chunk else 'a'
    header = first_chunk
    X_chunk.to_csv(X_file, mode=mode, index=False, header=header)
    X_ext_chunk.to_csv(X_ext_file, mode=mode, index=False, header=header)
    proxy_chunk.to_csv(proxy_file, mode=mode, index=False, header=header)
    y_chunk.to_csv(y_file, mode=mode, index=False, header=header)

    first_chunk = False

print("Data preprocessing complete. Binary loan labels generated using conservative strategy.")
print(f"{X_file} → main model features")
print(f"{X_ext_file} → with controversial features")
print(f"{proxy_file} → proxy group features")
print(f"{y_file} → binary target labels")


Analyzing chunk 1...
Analyzing chunk 2...
Analyzing chunk 3...
Analyzing chunk 4...
Analyzing chunk 5...
Data preprocessing complete. Binary loan labels generated using conservative strategy.
preprocess/data_features/model_features/original/X_features.csv → main model features
preprocess/data_features/model_features/original/X_features_with_controversial.csv → with controversial features
preprocess/data_features/proxy_features/proxy_features.csv → proxy group features
preprocess/data_features/binary_target_labels/target.csv → binary target labels


In [12]:
import pandas as pd

df = pd.read_csv('preprocess/data_features/model_features/original/X_features_with_controversial.csv')

# Quick field statistics
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str),
    'missing_rate': df.isnull().mean().round(4),
    'n_unique': df.nunique()
})

print(summary)


                                  column    dtype  missing_rate  n_unique
loan_amnt                      loan_amnt  float64        0.0000      1556
funded_amnt                  funded_amnt  float64        0.0000      1556
installment                  installment  float64        0.0000     83308
annual_inc                    annual_inc  float64        0.0000     64362
emp_length                    emp_length   object        0.0584        11
delinq_2yrs                  delinq_2yrs  float64        0.0000        31
inq_last_6mths            inq_last_6mths  float64        0.0000         9
open_acc                        open_acc  float64        0.0000        84
pub_rec                          pub_rec  float64        0.0000        37
revol_util                    revol_util  float64        0.0006      1373
total_acc                      total_acc  float64        0.0000       142
dti                                  dti  float64        0.0003      7067
purpose                          purpo

In [13]:
import pandas as pd
import os

input_path = 'preprocess/data_features/model_features/original/X_features_with_controversial.csv'
output_dir = 'preprocess/data_features/model_features/cleaned_onehot'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_path)

# Convert emp_length to numeric
def convert_emp_length(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if '< 1' in val:
        return 0
    elif '10+' in val:
        return 10
    elif 'n/a' in val:
        return None
    try:
        return int(val.split()[0])
    except:
        return None

df['emp_length'] = df['emp_length'].apply(convert_emp_length)
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

# Fill missing numeric values with median
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())

# Set category order to ensure stable one-hot encoding
category_order = {
    'purpose': [
        'credit_card', 'car', 'small_business', 'other', 'wedding',
        'debt_consolidation', 'home_improvement', 'major_purchase',
        'medical', 'vacation', 'house', 'moving', 'renewable_energy', 'educational'
    ],
    'verification_status': ['Not Verified', 'Verified', 'Source Verified']
}

for col, cats in category_order.items():
    df[col] = pd.Categorical(df[col], categories=cats)

# One-hot encode selected categorical columns
df_encoded = pd.get_dummies(df, columns=list(category_order.keys()), drop_first=True)

# Save the version with controversial features
X_ext_cleaned_path = os.path.join(output_dir, 'X_ext_cleaned.csv')
df_encoded.to_csv(X_ext_cleaned_path, index=False)

# Remove controversial features for fair model version
columns_to_remove = ['int_rate'] + [col for col in df_encoded.columns if col.startswith('verification_status_')]
df_clean = df_encoded.drop(columns=columns_to_remove)

X_cleaned_path = os.path.join(output_dir, 'X_cleaned.csv')
df_clean.to_csv(X_cleaned_path, index=False)

# Final message
print("Data cleaning and one-hot encoding completed.")
print(f"With controversial features: {X_ext_cleaned_path}")
print(f"Without controversial features: {X_cleaned_path}")


Data cleaning and one-hot encoding completed.
With controversial features: preprocess/data_features/model_features/cleaned_onehot\X_ext_cleaned.csv
Without controversial features: preprocess/data_features/model_features/cleaned_onehot\X_cleaned.csv


In [14]:
import pandas as pd
import os

input_path = 'preprocess/data_features/proxy_features/proxy_features.csv'
output_path = 'preprocess/data_features/proxy_features/proxy_cleaned.csv'
raw_proxy_path = 'preprocess/data_features/proxy_features/raw_proxy_cleaned.csv'

df = pd.read_csv(input_path)

# Drop high-cardinality variable zip_code
if 'zip_code' in df.columns:
    df = df.drop(columns=['zip_code'])

# Keep only selected proxy columns
keep_cols = ['home_ownership', 'application_type', 'addr_state']
df = df[keep_cols]

# Map addr_state to region
state_to_region = {
    'WA': 'West', 'OR': 'West', 'CA': 'West', 'NV': 'West', 'ID': 'West', 'MT': 'West', 'WY': 'West', 'UT': 'West', 'CO': 'West', 'AK': 'West', 'HI': 'West',
    'ND': 'Midwest', 'SD': 'Midwest', 'NE': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'IA': 'Midwest', 'MO': 'Midwest', 'WI': 'Midwest', 'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest',
    'TX': 'South', 'OK': 'South', 'AR': 'South', 'LA': 'South', 'KY': 'South', 'TN': 'South', 'MS': 'South', 'AL': 'South', 'WV': 'South', 'MD': 'South', 'DE': 'South', 'DC': 'South', 'VA': 'South', 'NC': 'South', 'SC': 'South', 'GA': 'South', 'FL': 'South',
    'NY': 'Northeast', 'NJ': 'Northeast', 'PA': 'Northeast', 'CT': 'Northeast', 'RI': 'Northeast', 'MA': 'Northeast', 'VT': 'Northeast', 'NH': 'Northeast', 'ME': 'Northeast'
}
df['region'] = df['addr_state'].map(state_to_region)

# Save raw proxy features before encoding
df.to_csv(raw_proxy_path, index=False)

# Set category orders for consistent encoding
category_orders = {
    'home_ownership': ['RENT', 'OWN', 'MORTGAGE', 'OTHER', 'NONE', 'ANY'],
    'application_type': ['INDIVIDUAL', 'JOINT'],
    'addr_state': sorted(df['addr_state'].dropna().unique().tolist()),
    'region': ['West', 'Midwest', 'South', 'Northeast']
}

for col, cats in category_orders.items():
    if col in df.columns:
        df[col] = pd.Categorical(df[col], categories=cats)

# One-hot encode categorical proxy columns
df_encoded = pd.get_dummies(df, columns=category_orders.keys(), drop_first=True)

# Save encoded proxy features
df_encoded.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Saved raw proxy features to: {raw_proxy_path}")
print(f"Saved cleaned and encoded proxy features to: {output_path}")
print(f"Output shape: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")


Saved raw proxy features to: preprocess/data_features/proxy_features/raw_proxy_cleaned.csv
Saved cleaned and encoded proxy features to: preprocess/data_features/proxy_features/proxy_cleaned.csv
Output shape: 1345350 rows, 59 columns


In [16]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

input_path = 'preprocess/data_features/model_features/cleaned_onehot/X_cleaned.csv'
target_path = 'preprocess/data_features/binary_target_labels/target.csv'
output_dir = 'preprocess/training_test_data/original'
os.makedirs(output_dir, exist_ok=True)

X = pd.read_csv(input_path)
y = pd.read_csv(target_path).squeeze()

# Set random seed
RANDOM_STATE = 17
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Save split data
X_train_path = os.path.join(output_dir, 'X_train.csv')
X_test_path = os.path.join(output_dir, 'X_test.csv')
y_train_path = os.path.join(output_dir, 'y_train.csv')
y_test_path = os.path.join(output_dir, 'y_test.csv')

X_train.to_csv(X_train_path, index=False)
X_test.to_csv(X_test_path, index=False)
y_train.to_csv(y_train_path, index=False)
y_test.to_csv(y_test_path, index=False)

# Print confirmation
print("Train/test split completed and saved successfully.")
print(f"X_train: {X_train_path}")
print(f"X_test:  {X_test_path}")
print(f"y_train: {y_train_path}")
print(f"y_test:  {y_test_path}")


Train/test split completed and saved successfully.
X_train: preprocess/training_test_data/original\X_train.csv
X_test:  preprocess/training_test_data/original\X_test.csv
y_train: preprocess/training_test_data/original\y_train.csv
y_test:  preprocess/training_test_data/original\y_test.csv


In [17]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

input_dir = 'preprocess/training_test_data/original'
output_dir = 'preprocess/training_test_data/scaled'
os.makedirs(output_dir, exist_ok=True)

X_train = pd.read_csv(os.path.join(input_dir, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(input_dir, 'X_test.csv'))

# Preserve column order
columns = X_train.columns.tolist()

# Fit scaler on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_path = os.path.join(output_dir, 'X_train_scaled.csv')
X_test_scaled_path = os.path.join(output_dir, 'X_test_scaled.csv')

pd.DataFrame(X_train_scaled, columns=columns).to_csv(X_train_scaled_path, index=False)
pd.DataFrame(X_test_scaled, columns=columns).to_csv(X_test_scaled_path, index=False)

print("Standardized datasets saved successfully.")
print(f"X_train_scaled: {X_train_scaled_path}")
print(f"X_test_scaled:  {X_test_scaled_path}")


Standardized datasets saved successfully.
X_train_scaled: preprocess/training_test_data/scaled\X_train_scaled.csv
X_test_scaled:  preprocess/training_test_data/scaled\X_test_scaled.csv


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

proxy_path = 'preprocess/data_features/proxy_features/raw_proxy_cleaned.csv'
y_all_path = 'preprocess/data_features/binary_target_labels/target.csv'
output_dir = 'preprocess/training_test_data/proxy_data'
os.makedirs(output_dir, exist_ok=True)

proxy_all = pd.read_csv(proxy_path)
y_all = pd.read_csv(y_all_path).squeeze()

# Reproduce train/test split using the same seed
proxy_train, _ = train_test_split(proxy_all, test_size=0.2, stratify=y_all, random_state=17)

# Save proxy training data
proxy_train_path = os.path.join(output_dir, 'proxy_train.csv')
proxy_train.to_csv(proxy_train_path, index=False)

print(f"proxy_train.csv saved to: {proxy_train_path}")


proxy_train.csv saved to: preprocess/training_test_data/proxy_data\proxy_train.csv


In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split

proxy_df = pd.read_csv('preprocess/data_features/proxy_features/raw_proxy_cleaned.csv')

# Load y (for stratified split consistency)
y_train = pd.read_csv('preprocess/training_test_data/original/y_train.csv')
y_test = pd.read_csv('preprocess/training_test_data/original/y_test.csv')
y = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

# Reproduce consistent test split
_, proxy_test = train_test_split(proxy_df, test_size=0.2, stratify=y, random_state=17)

# Save test proxy data
output_path = 'preprocess/training_test_data/proxy_data/proxy_test.csv'
proxy_test.to_csv(output_path, index=False)

print(f"proxy_test.csv saved successfully. Shape: {proxy_test.shape[0]} rows")


proxy_test.csv saved successfully. Shape: 269070 rows


In [35]:
import pandas as pd
import os

# Input paths
proxy_train_path = 'preprocess/training_test_data/proxy_data/proxy_train.csv'
proxy_test_path = 'preprocess/training_test_data/proxy_data/proxy_test.csv'

# Output paths
proxy_train_clean_path = 'preprocess/training_test_data/proxy_data/proxy_train_clean.csv'
proxy_test_clean_path = 'preprocess/training_test_data/proxy_data/proxy_test_clean.csv'
os.makedirs('preprocess/training_test_data/proxy_data', exist_ok=True)

# Define sensitive attribute and retained groups
sensitive_attr = 'home_ownership'
major_groups = ['MORTGAGE', 'RENT', 'OWN']

# Process training set
proxy_train = pd.read_csv(proxy_train_path)
proxy_train[sensitive_attr] = proxy_train[sensitive_attr].apply(
    lambda x: x if x in major_groups else 'OTHER'
)
proxy_train.to_csv(proxy_train_clean_path, index=False)

# Process test set
proxy_test = pd.read_csv(proxy_test_path)
proxy_test[sensitive_attr] = proxy_test[sensitive_attr].apply(
    lambda x: x if x in major_groups else 'OTHER'
)
proxy_test.to_csv(proxy_test_clean_path, index=False)

print(f"Cleaned proxy files saved to: {proxy_train_clean_path} and {proxy_test_clean_path}")


Cleaned proxy files saved to: preprocess/training_test_data/proxy_data/proxy_train_clean.csv and preprocess/training_test_data/proxy_data/proxy_test_clean.csv


In [8]:
import pandas as pd
import os
import joblib
import inspect
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve
import numpy as np

def find_best_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_index = f1_scores.argmax()
    return thresholds[best_index], precision[best_index], recall[best_index], f1_scores[best_index]

def train_and_evaluate_model(model, model_name: str, scaled: bool = False):
    # Input directories
    data_dir = 'preprocess/training_test_data/scaled' if scaled else 'preprocess/training_test_data/original'
    X_train = pd.read_csv(os.path.join(data_dir, 'X_train_scaled.csv' if scaled else 'X_train.csv'))
    X_test = pd.read_csv(os.path.join(data_dir, 'X_test_scaled.csv' if scaled else 'X_test.csv'))
    y_train = pd.read_csv('preprocess/training_test_data/original/y_train.csv').squeeze()
    y_test = pd.read_csv('preprocess/training_test_data/original/y_test.csv').squeeze()

    # Output root
    model_dir_root = 'model/original/initial_training'
    suffix = '_scaled' if scaled else ''

    # Fit model with sample weights if available
    fit_signature = inspect.signature(model.fit)
    if 'sample_weight' in fit_signature.parameters:
        sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
        model.fit(X_train, y_train, sample_weight=sample_weight)
        print(f"{model_name}: sample_weight applied.")
    else:
        model.fit(X_train, y_train)
        print(f"{model_name}: no sample_weight support.")

    # Save fitted model
    model_save_dir = os.path.join(model_dir_root, 'models')
    os.makedirs(model_save_dir, exist_ok=True)
    joblib.dump(model, os.path.join(model_save_dir, f'{model_name}{suffix}.pkl'))

    # Predict probabilities
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        raise ValueError(f"Model {model_name} does not support predict_proba")

    # Evaluation at threshold = 0.5
    y_pred_default = (y_prob >= 0.5).astype(int)
    metrics_0_5 = {
        'Threshold': 0.5,
        'AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob),
        'Precision': precision_score(y_test, y_pred_default),
        'Recall': recall_score(y_test, y_pred_default),
        'F1-score': f1_score(y_test, y_pred_default)
    }

    # Evaluation at best F1 threshold
    best_thres, best_prec, best_rec, best_f1 = find_best_threshold(y_test, y_prob)
    y_pred_best = (y_prob >= best_thres).astype(int)
    metrics_best = {
        'Threshold': best_thres,
        'AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob),
        'Precision': best_prec,
        'Recall': best_rec,
        'F1-score': best_f1
    }

    # Save predictions and metrics
    for thres_name, pred, metrics in [
        ("threshold_0.5", y_pred_default, metrics_0_5),
        ("threshold_best", y_pred_best, metrics_best)
    ]:
        out_dir = os.path.join(model_dir_root, thres_name, model_name + suffix)
        os.makedirs(out_dir, exist_ok=True)

        pd.DataFrame({
            'y_true': y_test,
            'y_pred': pred,
            'y_prob': y_prob
        }).to_csv(os.path.join(out_dir, 'test_predictions.csv'), index=False)

        pd.DataFrame([metrics]).to_csv(os.path.join(out_dir, 'evaluation_metrics.csv'), index=False)

        print(f"Saved predictions and metrics to: {out_dir}")


In [9]:
# Train and evaluate Logistic Regression on standardized data
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=17)
train_and_evaluate_model(lr, model_name='LogisticRegression', scaled=True)


LogisticRegression: sample_weight applied.
Saved predictions and metrics to: model/original/initial_training\threshold_0.5\LogisticRegression_scaled
Saved predictions and metrics to: model/original/initial_training\threshold_best\LogisticRegression_scaled


In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

# Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=17)
train_and_evaluate_model(rf, model_name='RandomForest', scaled=False)

# XGBoost: compute scale_pos_weight
y_train = pd.read_csv('preprocess/training_test_data/original/y_train.csv').squeeze()
neg, pos = np.bincount(y_train)
scale = neg / pos

xgb = XGBClassifier(
    eval_metric='logloss',
    scale_pos_weight=scale,
    random_state=17
)
train_and_evaluate_model(xgb, model_name='XGBoost', scaled=False)

# MLP (Neural Network)
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=17
)
train_and_evaluate_model(mlp, model_name='MLP', scaled=True)


RandomForest: sample_weight applied.
Saved predictions and metrics to: model/original/initial_training\threshold_0.5\RandomForest
Saved predictions and metrics to: model/original/initial_training\threshold_best\RandomForest
XGBoost: sample_weight applied.
Saved predictions and metrics to: model/original/initial_training\threshold_0.5\XGBoost
Saved predictions and metrics to: model/original/initial_training\threshold_best\XGBoost
MLP: no sample_weight support.
Saved predictions and metrics to: model/original/initial_training\threshold_0.5\MLP_scaled
Saved predictions and metrics to: model/original/initial_training\threshold_best\MLP_scaled


In [11]:
import pandas as pd
import os
import joblib
import matplotlib.pyplot as plt
import numpy as np

def rank_model_features(model_path, feature_path, output_folder, model_type='tree'):
    """
    Rank and visualize feature importances for a trained model.

    Parameters:
    - model_path: path to trained model (.pkl)
    - feature_path: path to input features CSV (must match model column order)
    - output_folder: directory to save ranking output
    - model_type: 'tree' (for XGBoost / Random Forest) or 'linear' (for Logistic Regression)
    """
    # Load model
    model = joblib.load(model_path)

    # Load feature names
    X = pd.read_csv(feature_path, nrows=1)
    feature_names = X.columns.tolist()

    # Get feature importance
    if model_type == 'tree':
        importances = model.feature_importances_
    elif model_type == 'linear':
        importances = np.abs(model.coef_.flatten())
    else:
        raise ValueError("model_type must be 'tree' or 'linear'")

    # Create importance dataframe
    df_rank = pd.DataFrame({
        'feature_name': feature_names,
        'importance': importances
    }).sort_values(by='importance', ascending=False).reset_index(drop=True)

    df_rank['rank'] = df_rank.index + 1

    # Save ranking
    os.makedirs(output_folder, exist_ok=True)
    df_rank.to_csv(os.path.join(output_folder, 'feature_ranking.csv'), index=False)

    # Plot top 20
    top_n = 20
    top_df = df_rank.head(top_n)
    plt.figure(figsize=(10, 6))
    plt.barh(top_df['feature_name'][::-1], top_df['importance'][::-1], color='skyblue')
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Features ({model_type})')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'feature_importance.png'))
    plt.close()

    print(f"Feature ranking and plot saved to: {output_folder}")


In [13]:
# Feature ranking for Logistic Regression
rank_model_features(
    model_path='model/original/initial_training/models/LogisticRegression_scaled.pkl',
    feature_path='preprocess/training_test_data/scaled/X_train_scaled.csv',
    output_folder='model/original/feature_ranking/LR_ranking',
    model_type='linear'
)

# Feature ranking for Random Forest
rank_model_features(
    model_path='model/original/initial_training/models/RandomForest.pkl',
    feature_path='preprocess/training_test_data/original/X_train.csv',
    output_folder='model/original/feature_ranking/RF_ranking',
    model_type='tree'
)

# Feature ranking for XGBoost
rank_model_features(
    model_path='model/original/initial_training/models/XGBoost.pkl',
    feature_path='preprocess/training_test_data/original/X_train.csv',
    output_folder='model/original/feature_ranking/XGB_ranking',
    model_type='tree'
)


Feature ranking and plot saved to: model/original/feature_ranking/LR_ranking
Feature ranking and plot saved to: model/original/feature_ranking/RF_ranking
Feature ranking and plot saved to: model/original/feature_ranking/XGB_ranking


In [14]:
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os

def rank_mlp_permutation(model_path, X_test_path, y_test_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Load model and data
    model = joblib.load(model_path)
    X_test = pd.read_csv(X_test_path)
    y_test = pd.read_csv(y_test_path).squeeze()

    # Compute permutation importance
    result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=17, n_jobs=-1)

    df_rank = pd.DataFrame({
        'feature_name': X_test.columns,
        'importance_mean': result.importances_mean,
        'importance_std': result.importances_std
    }).sort_values(by='importance_mean', ascending=False).reset_index(drop=True)

    df_rank['rank'] = df_rank.index + 1
    df_rank.to_csv(os.path.join(output_dir, 'feature_ranking.csv'), index=False)

    # Plot top 20
    top_n = 20
    plt.figure(figsize=(10, 6))
    plt.barh(df_rank['feature_name'][:top_n][::-1], df_rank['importance_mean'][:top_n][::-1], color='skyblue')
    plt.xlabel('Permutation Importance')
    plt.title('Top 20 Features (MLP - Permutation)')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'feature_importance.png'))
    plt.close()

    print(f"MLP permutation feature ranking saved to: {output_dir}")


In [16]:
# Feature ranking for MLP using permutation importance
rank_mlp_permutation(
    model_path='model/original/initial_training/models/MLP_scaled.pkl',
    X_test_path='preprocess/training_test_data/scaled/X_test_scaled.csv',
    y_test_path='preprocess/training_test_data/original/y_test.csv',
    output_dir='model/original/feature_ranking/MLP_ranking'
)


MLP permutation feature ranking saved to: model/original/feature_ranking/MLP_ranking


In [17]:
import pandas as pd
import os

def evaluate_model_fairness(
    pred_path,
    proxy_path,
    model_name,
    fairness_root='model/original/fairness_result',
    proxy_meta_path='preprocess/training_test_data/proxy_data/proxy_columns_used.csv',
    min_group_count=50
):
    """
    Evaluate group fairness metrics for a trained model using proxy features.

    Parameters:
    - pred_path: path to model prediction file (must include y_true, y_pred)
    - proxy_path: path to one-hot encoded proxy features
    - model_name: name of the model (used in result filenames)
    - fairness_root: root directory to save fairness result files
    - proxy_meta_path: path to save proxy column meta information
    - min_group_count: minimum samples per group to include in evaluation
    """

    # Load data
    pred_df = pd.read_csv(pred_path)
    proxy_df = pd.read_csv(proxy_path)
    assert len(pred_df) == len(proxy_df), "Prediction and proxy row count mismatch"

    df = pd.concat([pred_df, proxy_df], axis=1)

    # Filter proxy columns by group size
    group_counts = proxy_df.sum()
    valid_cols = group_counts[group_counts >= min_group_count].index.tolist()
    proxy_df_filtered = proxy_df[valid_cols]

    # Save meta info
    os.makedirs(os.path.dirname(proxy_meta_path), exist_ok=True)
    pd.DataFrame({
        'proxy_column': group_counts.index,
        'sample_count': group_counts.values,
        'kept': group_counts.index.isin(valid_cols)
    }).to_csv(proxy_meta_path, index=False)

    results = []

    for col in valid_cols:
        for group in [0, 1]:
            mask = df[col] == group
            y_true_group = df.loc[mask, 'y_true']
            y_pred_group = df.loc[mask, 'y_pred']

            if len(y_true_group) < min_group_count:
                continue

            tp = ((y_true_group == 1) & (y_pred_group == 1)).sum()
            fn = ((y_true_group == 1) & (y_pred_group == 0)).sum()
            tpr = tp / (tp + fn + 1e-6)
            pos_rate = (y_pred_group == 1).mean()

            results.append({
                'model': model_name,
                'proxy_feature': col,
                'group': group,
                'sample_size': len(y_true_group),
                'TPR (Equal Opportunity)': round(tpr, 4),
                'Positive Rate (Demographic Parity)': round(pos_rate, 4)
            })

    df_result = pd.DataFrame(results)

    # Save Equal Opportunity & Demographic Parity
    os.makedirs(os.path.join(fairness_root, 'EO', model_name), exist_ok=True)
    os.makedirs(os.path.join(fairness_root, 'DP', model_name), exist_ok=True)
    df_result.to_csv(
        os.path.join(fairness_root, 'EO', model_name, 'fairness_by_group.csv'),
        index=False
    )
    df_result.to_csv(
        os.path.join(fairness_root, 'DP', model_name, 'fairness_by_group.csv'),
        index=False
    )

    # Compute and save Disparate Impact
    di_summary = []
    for proxy in df_result['proxy_feature'].unique():
        sub = df_result[df_result['proxy_feature'] == proxy]
        max_pr = sub['Positive Rate (Demographic Parity)'].max()
        min_pr = sub['Positive Rate (Demographic Parity)'].min()
        di = round(min_pr / (max_pr + 1e-6), 4)
        di_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'disparate_impact': di
        })

    os.makedirs(os.path.join(fairness_root, 'DI', model_name), exist_ok=True)
    pd.DataFrame(di_summary).to_csv(
        os.path.join(fairness_root, 'DI', model_name, 'disparate_impact_summary.csv'),
        index=False
    )

    print(f"Fairness evaluation completed for model: {model_name}")


In [18]:
# Logistic Regression
evaluate_model_fairness(
    pred_path='model/original/initial_training/threshold_best/LogisticRegression_scaled/test_predictions.csv',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='LogisticRegression'
)

# Random Forest
evaluate_model_fairness(
    pred_path='model/original/initial_training/threshold_best/RandomForest/test_predictions.csv',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='RandomForest'
)

# XGBoost
evaluate_model_fairness(
    pred_path='model/original/initial_training/threshold_best/XGBoost/test_predictions.csv',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='XGBoost'
)

# MLP
evaluate_model_fairness(
    pred_path='model/original/initial_training/threshold_best/MLP_scaled/test_predictions.csv',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='MLP'
)



Fairness evaluation completed for model: LogisticRegression
Fairness evaluation completed for model: RandomForest
Fairness evaluation completed for model: XGBoost
Fairness evaluation completed for model: MLP


In [22]:
import pandas as pd
import os

def summarize_fairness_disparity(model_folders, output_path):
    """
    Summarize disparate impact results from multiple models into a single table.

    Parameters:
    - model_folders: dict of {model_name: folder containing disparate_impact_summary.csv}
    - output_path: path to save the combined summary table
    """
    summary = []

    for model_name, folder in model_folders.items():
        file_path = os.path.join(folder, 'disparate_impact_summary.csv')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            row = {'model': model_name}
            for _, r in df.iterrows():
                row[r['proxy_feature']] = r['disparate_impact']
            summary.append(row)
        else:
            print(f"File not found: {file_path}")

    df_summary = pd.DataFrame(summary).set_index('model')

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_summary.to_csv(output_path)

    print(f"Disparate impact summary saved to: {output_path}")
    return df_summary


In [23]:
model_dirs = {
    'LogisticRegression': 'model/original/fairness_result/DI/LogisticRegression',
    'RandomForest': 'model/original/fairness_result/DI/RandomForest',
    'XGBoost': 'model/original/fairness_result/DI/XGBoost',
    'MLP': 'model/original/fairness_result/DI/MLP'
}


fairness_summary_path = 'model/original/fairness_result/fairness_summary/disparity_impact_summary.csv'


df_fairness = summarize_fairness_disparity(model_dirs, fairness_summary_path)
print(df_fairness)


Disparate impact summary saved to: model/original/fairness_result/fairness_summary/disparity_impact_summary.csv
                    home_ownership_OWN  home_ownership_MORTGAGE  \
model                                                             
LogisticRegression              0.9936                   0.9962   
RandomForest                    0.9940                   0.9905   
XGBoost                         0.9955                   0.9902   
MLP                             0.9975                   0.9874   

                    home_ownership_ANY  addr_state_AL  addr_state_AR  \
model                                                                  
LogisticRegression              0.8405         0.9908         0.9918   
RandomForest                    0.8670         0.9815         0.9958   
XGBoost                         0.6845         0.9880         0.9845   
MLP                             0.8380         0.9941         0.9787   

                    addr_state_AZ  addr_state_CA  ad

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def plot_disparate_impact_heatmap(
    csv_path,
    output_path='model/original/fairness_result/fairness_summary/disparate_impact_heatmap.png'
):
    """
    Plot a heatmap of disparate impact values per model and proxy feature.

    Parameters:
    - csv_path: path to the disparity impact summary CSV file
    - output_path: path to save the generated heatmap image
    """
    df = pd.read_csv(csv_path).set_index('model')
    unfair_mask = df < 0.8

    n_cols = df.shape[1]
    n_rows = df.shape[0]
    cell_width = 1
    cell_height = 2

    plt.figure(figsize=(n_cols * cell_width, n_rows * cell_height + 2))

    ax = sns.heatmap(
        df.round(4),
        annot=True,
        fmt=".4f",
        cmap=sns.diverging_palette(150, 10, as_cmap=True),
        vmin=0.85,
        vmax=1.0,
        center=0.925,
        cbar_kws={'label': 'Disparate Impact'},
        linewidths=0.3,
        linecolor='gray',
        annot_kws={'size': 8}
    )

    # Highlight cells where DI < 0.8
    for i in range(n_rows):
        for j in range(n_cols):
            if unfair_mask.iloc[i, j]:
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='red', lw=2))

    plt.xticks(rotation=60, ha='right')
    plt.title('Disparate Impact per Model and Proxy Feature')
    plt.tight_layout()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"Heatmap saved to: {output_path}")


In [25]:
plot_disparate_impact_heatmap('model/original/fairness_result/fairness_summary/disparity_impact_summary.csv')


Heatmap saved to: model/original/fairness_result/fairness_summary/disparate_impact_heatmap.png


In [26]:
import pandas as pd
import os

def evaluate_equal_opportunity(fairness_by_group_path):
    """
    Evaluate Equal Opportunity gap (max TPR - min TPR) for each model and proxy feature.

    Parameters:
    - fairness_by_group_path: path to fairness_by_group.csv
    """
    df = pd.read_csv(fairness_by_group_path)

    eo_summary = []

    for (model, proxy), group_df in df.groupby(['model', 'proxy_feature']):
        tprs = group_df['TPR (Equal Opportunity)']
        max_tpr = tprs.max()
        min_tpr = tprs.min()
        gap = round(max_tpr - min_tpr, 4)

        eo_summary.append({
            'model': model,
            'proxy_feature': proxy,
            'TPR_max': round(max_tpr, 4),
            'TPR_min': round(min_tpr, 4),
            'TPR_gap': gap
        })

    df_eo = pd.DataFrame(eo_summary)

    # Save result to same folder
    output_path = os.path.join(
        os.path.dirname(fairness_by_group_path),
        'equal_opportunity_gap.csv'
    )

    df_eo.to_csv(output_path, index=False)
    print(f"Equal Opportunity gap summary saved to: {output_path}")
    return df_eo


In [29]:
# Equal Opportunity evaluation for Logistic Regression
evaluate_equal_opportunity(
    fairness_by_group_path='model/original/fairness_result/EO/LogisticRegression/fairness_by_group.csv'
)

# Equal Opportunity evaluation for MLP
evaluate_equal_opportunity(
    fairness_by_group_path='model/original/fairness_result/EO/MLP/fairness_by_group.csv'
)

# Equal Opportunity evaluation for Random Forest
evaluate_equal_opportunity(
    fairness_by_group_path='model/original/fairness_result/EO/RandomForest/fairness_by_group.csv'
)

# Equal Opportunity evaluation for XGBoost
evaluate_equal_opportunity(
    fairness_by_group_path='model/original/fairness_result/EO/XGBoost/fairness_by_group.csv'
)


Equal Opportunity gap summary saved to: model/original/fairness_result/EO/LogisticRegression\equal_opportunity_gap.csv
Equal Opportunity gap summary saved to: model/original/fairness_result/EO/MLP\equal_opportunity_gap.csv
Equal Opportunity gap summary saved to: model/original/fairness_result/EO/RandomForest\equal_opportunity_gap.csv
Equal Opportunity gap summary saved to: model/original/fairness_result/EO/XGBoost\equal_opportunity_gap.csv


,model,proxy_feature,TPR_max,TPR_min,TPR_gap
0,XGBoost,addr_state_AL,0.6606,0.6370,0.0236
1,XGBoost,addr_state_AR,0.6375,0.6110,0.0265
2,XGBoost,addr_state_AZ,0.6382,0.6007,0.0375
3,XGBoost,addr_state_CA,0.6376,0.6356,0.0020
4,XGBoost,addr_state_CO,0.6376,0.6259,0.0117
5,XGBoost,addr_state_CT,0.6374,0.6296,0.0078
6,XGBoost,addr_state_DC,0.6375,0.5822,0.0553
7,XGBoost,addr_state_DE,0.6712,0.6372,0.0340
8,XGBoost,addr_state_FL,0.6397,0.6371,0.0026
9,XGBoost,addr_state_GA,0.6373,0.6371,0.0002


In [30]:
import pandas as pd
import os

def summarize_equal_opportunity(model_paths: dict, output_path: str):
    """
    Summarize Equal Opportunity (TPR gap) results from multiple models.

    Parameters:
    - model_paths: dict {model_name: path to EO result folder containing equal_opportunity_gap.csv}
    - output_path: path to save the summary pivot table
    """
    all_rows = []

    for model_name, path in model_paths.items():
        file_path = os.path.join(path, 'equal_opportunity_gap.csv')
        if not os.path.exists(file_path):
            print(f"File not found, skipped: {file_path}")
            continue

        df = pd.read_csv(file_path)
        for _, row in df.iterrows():
            all_rows.append({
                'model': model_name,
                'proxy_feature': row['proxy_feature'],
                'TPR_gap': row['TPR_gap']
            })

    df_summary = pd.DataFrame(all_rows)
    df_pivot = df_summary.pivot(index='model', columns='proxy_feature', values='TPR_gap')

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_pivot.to_csv(output_path)

    print(f"Equal Opportunity gap summary saved to: {output_path}")
    return df_pivot


In [31]:
# Define correct fairness result paths for each model
model_fairness_paths = {
    'LogisticRegression': 'model/original/fairness_result/EO/LogisticRegression',
    'RandomForest': 'model/original/fairness_result/EO/RandomForest',
    'XGBoost': 'model/original/fairness_result/EO/XGBoost',
    'MLP': 'model/original/fairness_result/EO/MLP'
}

# Summarize TPR gaps and save to standard location
summarize_equal_opportunity(
    model_paths=model_fairness_paths,
    output_path='model/original/fairness_result/fairness_summary/equal_opportunity_summary.csv'
)


Equal Opportunity gap summary saved to: model/original/fairness_result/fairness_summary/equal_opportunity_summary.csv


proxy_feature,addr_state_AL,addr_state_AR,addr_state_AZ,addr_state_CA,addr_state_CO,addr_state_CT,addr_state_DC,addr_state_DE,addr_state_FL,addr_state_GA,...,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_OWN,region_Midwest,region_Northeast,region_South
model,,,,,,,,,,,,,,,,,,,,,
LogisticRegression,0.0106,0.0047,0.0257,0.0075,0.0288,0.0236,0.0429,0.0052,0.0020,0.0095,...,0.0069,0.0165,0.0209,0.0227,0.0551,0.0059,0.0039,0.0112,0.0125,0.0048
MLP,0.0306,0.0233,0.0266,0.0006,0.0104,0.0279,0.0498,0.0327,0.0022,0.0037,...,0.0011,0.0301,0.0283,0.0410,0.2033,0.0071,0.0034,0.0058,0.0079,0.0040
RandomForest,0.0351,0.0159,0.0161,0.0045,0.0208,0.0222,0.0058,0.0285,0.0030,0.0141,...,0.0140,0.0074,0.0086,0.0619,0.2279,0.0055,0.0073,0.0068,0.0110,0.0097
XGBoost,0.0236,0.0265,0.0375,0.0020,0.0117,0.0078,0.0553,0.0340,0.0026,0.0002,...,0.0028,0.0045,0.0127,0.0172,0.0770,0.0038,0.0061,0.0051,0.0002,0.0022


In [32]:
import pandas as pd
import os

def compute_demographic_parity(pred_path, proxy_path, model_name):
    """
    Compute Demographic Parity Gap for each proxy variable.

    Parameters:
    - pred_path: path to the folder containing test_predictions.csv
    - proxy_path: path to proxy test CSV
    - model_name: model name used for output folder
    """
    preds = pd.read_csv(os.path.join(pred_path, 'test_predictions.csv'))
    proxies = pd.read_csv(proxy_path)

    assert 'y_pred' in preds.columns, "Missing 'y_pred' column in predictions"

    df = pd.concat([proxies.reset_index(drop=True), preds['y_pred']], axis=1)

    results = []

    for column in proxies.columns:
        group_rates = df.groupby(column)['y_pred'].mean()
        gap = group_rates.max() - group_rates.min()
        result = {
            'proxy_variable': column,
            'min_positive_rate': round(group_rates.min(), 4),
            'max_positive_rate': round(group_rates.max(), 4),
            'dp_gap': round(gap, 4)
        }
        results.append(result)

    summary_df = pd.DataFrame(results)

    out_dir = os.path.join('model/original/fairness_result/DP', model_name)
    os.makedirs(out_dir, exist_ok=True)

    outpath = os.path.join(out_dir, 'demographic_parity_summary.csv')
    summary_df.to_csv(outpath, index=False)

    print(f"Demographic Parity summary saved to: {outpath}")


In [33]:
# Logistic Regression
compute_demographic_parity(
    pred_path='model/original/initial_training/threshold_best/LogisticRegression_scaled',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='LogisticRegression'
)

# Random Forest
compute_demographic_parity(
    pred_path='model/original/initial_training/threshold_best/RandomForest',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='RandomForest'
)

# XGBoost
compute_demographic_parity(
    pred_path='model/original/initial_training/threshold_best/XGBoost',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='XGBoost'
)

# MLP
compute_demographic_parity(
    pred_path='model/original/initial_training/threshold_best/MLP_scaled',
    proxy_path='preprocess/training_test_data/proxy_data/proxy_test.csv',
    model_name='MLP'
)


Demographic Parity summary saved to: model/original/fairness_result/DP\LogisticRegression\demographic_parity_summary.csv
Demographic Parity summary saved to: model/original/fairness_result/DP\RandomForest\demographic_parity_summary.csv
Demographic Parity summary saved to: model/original/fairness_result/DP\XGBoost\demographic_parity_summary.csv
Demographic Parity summary saved to: model/original/fairness_result/DP\MLP\demographic_parity_summary.csv


In [34]:
import pandas as pd
import os

# Model names and corresponding DP summary paths
models = {
    "LogisticRegression": "model/original/fairness_result/DP/LogisticRegression/demographic_parity_summary.csv",
    "MLP": "model/original/fairness_result/DP/MLP/demographic_parity_summary.csv",
    "RandomForest": "model/original/fairness_result/DP/RandomForest/demographic_parity_summary.csv",
    "XGBoost": "model/original/fairness_result/DP/XGBoost/demographic_parity_summary.csv"
}

# Store all results
dp_results = []

# Load and combine
for model_name, path in models.items():
    df = pd.read_csv(path)
    df['model'] = model_name
    dp_results.append(df)

dp_combined = pd.concat(dp_results, ignore_index=True)

# Save the combined summary
output_path = "model/original/fairness_result/fairness_summary/demographic_parity_all_models.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
dp_combined.to_csv(output_path, index=False)

print("Demographic Parity summary for all models saved to:", output_path)


Demographic Parity summary for all models saved to: model/original/fairness_result/fairness_summary/demographic_parity_all_models.csv


In [35]:
import pandas as pd
import os

def rank_proxy_fairness_gaps(model_name):
    """
    Compute and rank DP and EO gaps for each proxy feature in a model.
    Save result as CSV.
    """
    input_path = f"model/original/fairness_result/EO/{model_name}/fairness_by_group.csv"
    output_dir = f"model/original/fairness_result/proxy_ranking/{model_name}"
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_path)
    results = []

    for proxy in df['proxy_feature'].unique():
        sub = df[df['proxy_feature'] == proxy]
        pr_gap = sub['Positive Rate (Demographic Parity)'].max() - sub['Positive Rate (Demographic Parity)'].min()
        tpr_gap = sub['TPR (Equal Opportunity)'].max() - sub['TPR (Equal Opportunity)'].min()
        results.append({
            'proxy_feature': proxy,
            'positive_rate_gap': round(pr_gap, 4),
            'tpr_gap': round(tpr_gap, 4)
        })

    result_df = pd.DataFrame(results).sort_values(by='positive_rate_gap', ascending=False).reset_index(drop=True)
    output_path = os.path.join(output_dir, 'proxy_fairness_ranking.csv')
    result_df.to_csv(output_path, index=False)
    print(f"Fairness gap ranking saved for {model_name} → {output_path}")
    return result_df


In [36]:
rank_proxy_fairness_gaps("LogisticRegression")
rank_proxy_fairness_gaps("RandomForest")
rank_proxy_fairness_gaps("XGBoost")
rank_proxy_fairness_gaps("MLP")


Fairness gap ranking saved for LogisticRegression → model/original/fairness_result/proxy_ranking/LogisticRegression\proxy_fairness_ranking.csv
Fairness gap ranking saved for RandomForest → model/original/fairness_result/proxy_ranking/RandomForest\proxy_fairness_ranking.csv
Fairness gap ranking saved for XGBoost → model/original/fairness_result/proxy_ranking/XGBoost\proxy_fairness_ranking.csv
Fairness gap ranking saved for MLP → model/original/fairness_result/proxy_ranking/MLP\proxy_fairness_ranking.csv


,proxy_feature,positive_rate_gap,tpr_gap
0,home_ownership_ANY,0.0780,0.2033
1,addr_state_ND,0.0321,0.0468
2,addr_state_ID,0.0286,0.0225
3,addr_state_VT,0.0235,0.0187
4,addr_state_HI,0.0189,0.0326
5,addr_state_ME,0.0166,0.0145
6,addr_state_SD,0.0151,0.0284
7,addr_state_WY,0.0142,0.0410
8,addr_state_KS,0.0132,0.0153
9,addr_state_NE,0.0131,0.0698


In [36]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

# --- Path configuration ---
X_path = 'preprocess/training_test_data/original/X_train.csv'
y_path = 'preprocess/training_test_data/original/y_train.csv'
proxy_path = 'preprocess/training_test_data/proxy_data/proxy_train_clean.csv'
output_path = 'model/reweighed/reweighing_setting/sample_weight.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# --- Load data ---
X = pd.read_csv(X_path)
y = pd.read_csv(y_path)
proxy = pd.read_csv(proxy_path)

# --- Merge into full dataframe ---
df_full = X.copy()
df_full['label'] = y.values

# Set sensitive attribute
sensitive_attr = 'home_ownership'
df_full[sensitive_attr] = proxy[sensitive_attr].values

# Encode sensitive attribute to numeric
df_full[sensitive_attr] = LabelEncoder().fit_transform(df_full[sensitive_attr].astype(str))

# Select only features + proxy + label
cols = X.columns.tolist() + [sensitive_attr, 'label']
df_used = df_full[cols]

# Ensure numeric format
df_used = df_used.apply(pd.to_numeric, errors='coerce')
df_used.dropna(inplace=True)

# Construct AIF360 dataset
aif_data = BinaryLabelDataset(
    df=df_used,
    label_names=['label'],
    protected_attribute_names=[sensitive_attr]
)

# Identify groups
vals = df_used[sensitive_attr].unique()
privileged_val = vals[0]
unprivileged_vals = [v for v in vals if v != privileged_val]

# Apply Reweighing
RW = Reweighing(
    privileged_groups=[{sensitive_attr: privileged_val}],
    unprivileged_groups=[{sensitive_attr: v} for v in unprivileged_vals]
)
RW.fit(aif_data)
reweighed_data = RW.transform(aif_data)

# Save weights
sample_weight = reweighed_data.instance_weights
pd.DataFrame({'sample_weight': sample_weight}).to_csv(output_path, index=False)

print(f"Reweighing sample weights saved to: {output_path}")


Reweighing sample weights saved to: model/reweighed/reweighing_setting/sample_weight.csv


In [37]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, average_precision_score, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

def find_best_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    thresholds = np.append(thresholds, 1.0)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_index = f1_scores.argmax()
    return thresholds[best_index], precision[best_index], recall[best_index], f1_scores[best_index]

def train_reweighed_step1(model, model_name, scaled=True, sensitive_attr='home_ownership'):
    # --- Paths ---
    base_dir = 'preprocess/training_test_data/scaled' if scaled else 'preprocess/training_test_data/original'
    X_train = pd.read_csv(os.path.join(base_dir, 'X_train_scaled.csv' if scaled else 'X_train.csv'))
    X_test = pd.read_csv(os.path.join(base_dir, 'X_test_scaled.csv' if scaled else 'X_test.csv'))
    y_train = pd.read_csv('preprocess/training_test_data/original/y_train.csv').squeeze()
    y_test = pd.read_csv('preprocess/training_test_data/original/y_test.csv').squeeze()

    proxy_train = pd.read_csv('preprocess/training_test_data/proxy_data/proxy_train_clean.csv')
    proxy_test = pd.read_csv('preprocess/training_test_data/proxy_data/proxy_test_clean.csv')

    # --- AIF360 dataset ---
    df_train = X_train.copy()
    df_train['label'] = y_train.values
    df_train[sensitive_attr] = proxy_train[sensitive_attr].astype(str)
    df_train[sensitive_attr] = LabelEncoder().fit_transform(df_train[sensitive_attr])

    dataset = BinaryLabelDataset(
        df=df_train,
        label_names=['label'],
        protected_attribute_names=[sensitive_attr]
    )

    vals = df_train[sensitive_attr].unique()
    RW = Reweighing(
        privileged_groups=[{sensitive_attr: vals[0]}],
        unprivileged_groups=[{sensitive_attr: v} for v in vals[1:]]
    )
    RW.fit(dataset)
    dataset_rw = RW.transform(dataset)
    weights = dataset_rw.instance_weights

    # --- Train model ---
    out_dir = f'model/reweighed/training/{model_name}_scaled' if scaled else f'model/reweighed/training/{model_name}'
    os.makedirs(out_dir, exist_ok=True)
    model.fit(X_train, y_train, sample_weight=weights)
    joblib.dump(model, os.path.join(out_dir, f'{model_name}.pkl'))

    # --- Predictions & evaluation ---
    y_prob = model.predict_proba(X_test)[:, 1]
    thres_best, prec_b, rec_b, f1_b = find_best_threshold(y_test, y_prob)
    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_best = (y_prob >= thres_best).astype(int)

    for tag, pred, thres in [('threshold_0.5', y_pred_05, 0.5), ('threshold_best', y_pred_best, thres_best)]:
        path = os.path.join(out_dir, tag)
        os.makedirs(path, exist_ok=True)
        pd.DataFrame({'y_true': y_test, 'y_pred': pred, 'y_prob': y_prob}).to_csv(f'{path}/test_predictions.csv', index=False)
        metrics = {
            'Threshold': thres,
            'AUC': roc_auc_score(y_test, y_prob),
            'PR-AUC': average_precision_score(y_test, y_prob),
            'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred),
            'F1-score': f1_score(y_test, pred)
        }
        pd.DataFrame([metrics]).to_csv(f'{path}/evaluation_metrics.csv', index=False)

    # --- Fairness grouping ---
    df_fair = pd.concat([
        pd.Series(y_test).reset_index(drop=True),
        pd.Series(y_pred_best).reset_index(drop=True),
        proxy_test.reset_index(drop=True)
    ], axis=1)
    df_fair.columns = ['y_true', 'y_pred'] + list(proxy_test.columns)

    results = []
    for col in proxy_test.columns:
        for group in df_fair[col].dropna().unique():
            subset = df_fair[df_fair[col] == group]
            if len(subset) < 50:
                continue
            tp = ((subset['y_true'] == 1) & (subset['y_pred'] == 1)).sum()
            fn = ((subset['y_true'] == 1) & (subset['y_pred'] == 0)).sum()
            tpr = tp / (tp + fn + 1e-6)
            pos_rate = (subset['y_pred'] == 1).mean()
            results.append({
                'model': model_name,
                'proxy_feature': col,
                'group': group,
                'sample_size': len(subset),
                'TPR (Equal Opportunity)': round(tpr, 4),
                'Positive Rate (Demographic Parity)': round(pos_rate, 4)
            })

    df_group = pd.DataFrame(results)
    os.makedirs(os.path.join(out_dir, 'fairness_result'), exist_ok=True)
    df_group.to_csv(os.path.join(out_dir, 'fairness_result/fairness_by_group.csv'), index=False)

    print(f"{model_name} training with reweighing completed. Outputs saved to: {out_dir}")


In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import pandas as pd

# Load training labels and compute class ratio for XGBoost
y_train = pd.read_csv('preprocess/training_test_data/original/y_train.csv').squeeze()
scale = (y_train == 0).sum() / (y_train == 1).sum()

# Reweighed Logistic Regression (standardized features)
train_reweighed_step1(
    model=LogisticRegression(max_iter=3000, random_state=17),
    model_name='LogisticRegression',
    scaled=True
)

# Reweighed Random Forest (original features)
train_reweighed_step1(
    model=RandomForestClassifier(n_estimators=100, random_state=17),
    model_name='RandomForest',
    scaled=False
)

# Reweighed XGBoost (original features, class weight balanced)
train_reweighed_step1(
    model=XGBClassifier(scale_pos_weight=scale, eval_metric='logloss', random_state=17),
    model_name='XGBoost',
    scaled=False
)


LogisticRegression training with reweighing completed. Outputs saved to: model/reweighed/training/LogisticRegression_scaled
RandomForest training with reweighing completed. Outputs saved to: model/reweighed/training/RandomForest
XGBoost training with reweighing completed. Outputs saved to: model/reweighed/training/XGBoost


In [39]:
# Reweighed MLP (standardized features)
train_reweighed_step1(
    model=MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=17),
    model_name='MLP',
    scaled=True
)

MLP training with reweighing completed. Outputs saved to: model/reweighed/training/MLP_scaled


In [40]:
import pandas as pd
import os

def compute_fairness_metrics_reweighed(model_name, scaled=True):
    # Input path: where the trained reweighed model's fairness_by_group is stored
    train_dir = f'model/reweighed/training/{model_name}_scaled' if scaled else f'model/reweighed/training/{model_name}'
    input_fbg = os.path.join(train_dir, 'fairness_result/fairness_by_group.csv')
    df = pd.read_csv(input_fbg)

    # Output directory
    result_dir = f'model/reweighed/fairness_result/{model_name}'
    os.makedirs(result_dir, exist_ok=True)

    # --- Equal Opportunity Gap ---
    eo_summary = []
    for proxy in df['proxy_feature'].unique():
        sub = df[df['proxy_feature'] == proxy]
        max_tpr = sub['TPR (Equal Opportunity)'].max()
        min_tpr = sub['TPR (Equal Opportunity)'].min()
        eo_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'TPR_max': round(max_tpr, 4),
            'TPR_min': round(min_tpr, 4),
            'TPR_gap': round(max_tpr - min_tpr, 4)
        })
    pd.DataFrame(eo_summary).to_csv(os.path.join(result_dir, 'equal_opportunity_gap.csv'), index=False)

    # --- Disparate Impact ---
    di_summary = []
    for proxy in df['proxy_feature'].unique():
        sub = df[df['proxy_feature'] == proxy]
        max_pos = sub['Positive Rate (Demographic Parity)'].max()
        min_pos = sub['Positive Rate (Demographic Parity)'].min()
        di_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'disparate_impact': round(min_pos / (max_pos + 1e-6), 4)
        })
    pd.DataFrame(di_summary).to_csv(os.path.join(result_dir, 'disparate_impact_summary.csv'), index=False)

    # --- Demographic Parity Gap ---
    dp_summary = []
    for proxy in df['proxy_feature'].unique():
        sub = df[df['proxy_feature'] == proxy]
        max_pos = sub['Positive Rate (Demographic Parity)'].max()
        min_pos = sub['Positive Rate (Demographic Parity)'].min()
        dp_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'max_positive_rate': round(max_pos, 4),
            'min_positive_rate': round(min_pos, 4),
            'dp_gap': round(max_pos - min_pos, 4)
        })
    pd.DataFrame(dp_summary).to_csv(os.path.join(result_dir, 'demographic_parity_summary.csv'), index=False)

    print(f"{model_name} reweighed fairness metrics saved to: {result_dir}")


# Batch process all models
models = [
    ('LogisticRegression', True),
    ('RandomForest', False),
    ('XGBoost', False),
    ('MLP', True)
]

for model_name, scaled in models:
    compute_fairness_metrics_reweighed(model_name, scaled)


LogisticRegression reweighed fairness metrics saved to: model/reweighed/fairness_result/LogisticRegression
RandomForest reweighed fairness metrics saved to: model/reweighed/fairness_result/RandomForest
XGBoost reweighed fairness metrics saved to: model/reweighed/fairness_result/XGBoost
MLP reweighed fairness metrics saved to: model/reweighed/fairness_result/MLP


In [41]:
import pandas as pd
import os

# Model fairness output paths (reweighed)
model_dirs = {
    'LogisticRegression': 'model/reweighed/fairness_result/LogisticRegression',
    'RandomForest': 'model/reweighed/fairness_result/RandomForest',
    'XGBoost': 'model/reweighed/fairness_result/XGBoost',
    'MLP': 'model/reweighed/fairness_result/MLP'
}

# Output directory for fairness summaries
summary_dir = 'model/reweighed/fairness_result/fairness_summary'
os.makedirs(summary_dir, exist_ok=True)

# --- 1. Disparate Impact Summary ---
di_rows = []
for model, path in model_dirs.items():
    file = os.path.join(path, 'disparate_impact_summary.csv')
    if os.path.exists(file):
        df = pd.read_csv(file)
        row = {'model': model}
        for _, r in df.iterrows():
            row[r['proxy_feature']] = r['disparate_impact']
        di_rows.append(row)

df_di = pd.DataFrame(di_rows).set_index('model')
df_di.to_csv(os.path.join(summary_dir, 'disparity_impact_summary.csv'))

# --- 2. Equal Opportunity Gap Summary ---
eo_rows = []
for model, path in model_dirs.items():
    file = os.path.join(path, 'equal_opportunity_gap.csv')
    if os.path.exists(file):
        df = pd.read_csv(file)
        row = {'model': model}
        for _, r in df.iterrows():
            row[r['proxy_feature']] = r['TPR_gap']
        eo_rows.append(row)

df_eo = pd.DataFrame(eo_rows).set_index('model')
df_eo.to_csv(os.path.join(summary_dir, 'equal_opportunity_summary.csv'))

# --- 3. Demographic Parity Full Table ---
dp_frames = []
for model, path in model_dirs.items():
    file = os.path.join(path, 'demographic_parity_summary.csv')
    if os.path.exists(file):
        df = pd.read_csv(file)
        df['model'] = model
        dp_frames.append(df)

if dp_frames:
    df_dp = pd.concat(dp_frames, ignore_index=True)
    df_dp.to_csv(os.path.join(summary_dir, 'demographic_parity_all_models.csv'), index=False)

print("Reweighed model fairness summary tables saved to:", summary_dir)


Reweighed model fairness summary tables saved to: model/reweighed/fairness_result/fairness_summary


In [42]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def plot_disparate_impact_heatmap(csv_path, output_path):
    df = pd.read_csv(csv_path).set_index('model')
    unfair_mask = df < 0.8

    n_cols = df.shape[1]
    n_rows = df.shape[0]
    cell_width = 1
    cell_height = 2

    plt.figure(figsize=(n_cols * cell_width, n_rows * cell_height + 2))
    ax = sns.heatmap(
        df.round(4),
        annot=True,
        fmt=".4f",
        cmap=sns.diverging_palette(150, 10, as_cmap=True),
        vmin=0.85,
        vmax=1.0,
        center=0.925,
        cbar_kws={'label': 'Disparate Impact'},
        linewidths=0.3,
        linecolor='gray',
        annot_kws={'size': 8}
    )

    for i in range(n_rows):
        for j in range(n_cols):
            if unfair_mask.iloc[i, j]:
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='red', lw=2))

    plt.xticks(rotation=60, ha='right')
    plt.title('Disparate Impact per Reweighed Model and Proxy Feature')
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"Disparate Impact heatmap saved to: {output_path}")

# Corrected paths
csv_path = 'model/reweighed/fairness_result/fairness_summary/disparity_impact_summary.csv'
img_path = 'model/reweighed/fairness_result/pictures/disparate_impact_heatmap.png'

plot_disparate_impact_heatmap(csv_path, img_path)


Disparate Impact heatmap saved to: model/reweighed/fairness_result/pictures/disparate_impact_heatmap.png


In [43]:
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, average_precision_score, precision_recall_curve
)
from sklearn.model_selection import train_test_split

def train_fairness_constraints_logreg():
    base_dir = 'preprocess/training_test_data/scaled'
    out_dir = 'model/fairness_constraints/LogisticRegression_scaled'
    os.makedirs(out_dir, exist_ok=True)

    # Load data
    X_train = pd.read_csv(f'{base_dir}/X_train_scaled.csv')
    X_test = pd.read_csv(f'{base_dir}/X_test_scaled.csv')

    def load_label(path):
        df = pd.read_csv(path)
        if isinstance(df, pd.Series):
            return df
        elif df.shape[1] == 1:
            return df.iloc[:, 0]
        else:
            cols = [c for c in df.columns if 'label' in c.lower()]
            assert len(cols) == 1, f"Failed to identify label column: {df.columns.tolist()}"
            return df[cols[0]]

    y_train = load_label('preprocess/training_test_data/original/y_train.csv')
    y_test = load_label('preprocess/training_test_data/original/y_test.csv')

    # Load sensitive attribute (cleaned version)
    A_train = pd.read_csv('preprocess/training_test_data/proxy_data/proxy_train_clean.csv')['home_ownership'].astype(str)

    # Train fairness-constrained model
    estimator = LogisticRegression(max_iter=3000, random_state=17)
    mitigator = ExponentiatedGradient(
        estimator=estimator,
        constraints=EqualizedOdds()
    )
    mitigator.fit(X_train, y_train, sensitive_features=A_train)

    # Predict probabilities
    pmf = mitigator._pmf_predict(X_test)
    assert pmf.ndim == 2 and pmf.shape[1] == 2, f"y_prob must be (n, 2), got {pmf.shape}"
    y_prob = pmf[:, 1]

    # Threshold selection (best F1)
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    thresholds = list(thresholds) + [1.0]
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = f1_scores.argmax()
    threshold_best = thresholds[best_idx]

    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_best = (y_prob >= threshold_best).astype(int)

    for tag, y_pred, thres in [('threshold_0.5', y_pred_05, 0.5), ('threshold_best', y_pred_best, threshold_best)]:
        path = os.path.join(out_dir, tag)
        os.makedirs(path, exist_ok=True)

        pd.DataFrame({
            'y_true': y_test.values,
            'y_pred': y_pred,
            'y_prob': y_prob
        }).to_csv(os.path.join(path, 'test_predictions.csv'), index=False)

        metrics = {
            'Threshold': thres,
            'AUC': roc_auc_score(y_test, y_prob),
            'PR-AUC': average_precision_score(y_test, y_prob),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1-score': f1_score(y_test, y_pred)
        }
        pd.DataFrame([metrics]).to_csv(os.path.join(path, 'evaluation_metrics.csv'), index=False)

    print(f"Fairness-constrained Logistic Regression outputs saved to: {out_dir}")


In [45]:
train_fairness_constraints_logreg()


D:\APP\Projects\JupyterProject\.venv310\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\APP\Projects\JupyterProject\.venv310\lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:185: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the cavea

Fairness-constrained Logistic Regression outputs saved to: model/fairness_constraints/LogisticRegression_scaled


In [49]:
import pandas as pd
import os

def evaluate_fairness_constraints_logreg():
    model_name = 'LogisticRegression'
    base_dir = 'model/fairness_constraints/LogisticRegression_scaled'
    result_dir = os.path.join(base_dir, 'fairness_result')
    os.makedirs(result_dir, exist_ok=True)

    # Load predictions (best F1)
    preds = pd.read_csv(os.path.join(base_dir, 'threshold_best', 'test_predictions.csv'))
    y_test = preds['y_true']
    y_pred = preds['y_pred']

    # Load proxy test set (cleaned version)
    proxy_test = pd.read_csv('preprocess/training_test_data/proxy_data/proxy_test_clean.csv')

    df = pd.concat([
        y_test.reset_index(drop=True),
        y_pred.reset_index(drop=True),
        proxy_test.reset_index(drop=True)
    ], axis=1)
    df.columns = ['y_true', 'y_pred'] + list(proxy_test.columns)

    # --- fairness_by_group.csv ---
    records = []
    for col in proxy_test.columns:
        for group in df[col].dropna().unique():
            subset = df[df[col] == group]
            if len(subset) < 50:
                continue
            tp = ((subset['y_true'] == 1) & (subset['y_pred'] == 1)).sum()
            fn = ((subset['y_true'] == 1) & (subset['y_pred'] == 0)).sum()
            tpr = tp / (tp + fn + 1e-6)
            pos_rate = (subset['y_pred'] == 1).mean()
            records.append({
                'model': model_name,
                'proxy_feature': col,
                'group': group,
                'sample_size': len(subset),
                'TPR (Equal Opportunity)': round(tpr, 4),
                'Positive Rate (Demographic Parity)': round(pos_rate, 4)
            })

    df_fair = pd.DataFrame(records)
    df_fair.to_csv(os.path.join(result_dir, 'fairness_by_group.csv'), index=False)

    # --- disparate_impact_summary.csv ---
    di_summary = []
    for proxy in df_fair['proxy_feature'].unique():
        sub = df_fair[df_fair['proxy_feature'] == proxy]
        max_pos = sub['Positive Rate (Demographic Parity)'].max()
        min_pos = sub['Positive Rate (Demographic Parity)'].min()
        di = round(min_pos / (max_pos + 1e-6), 4)
        di_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'disparate_impact': di
        })
    pd.DataFrame(di_summary).to_csv(os.path.join(result_dir, 'disparate_impact_summary.csv'), index=False)

    # --- equal_opportunity_gap.csv ---
    eo_summary = []
    for proxy in df_fair['proxy_feature'].unique():
        sub = df_fair[df_fair['proxy_feature'] == proxy]
        max_tpr = sub['TPR (Equal Opportunity)'].max()
        min_tpr = sub['TPR (Equal Opportunity)'].min()
        eo_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'TPR_max': round(max_tpr, 4),
            'TPR_min': round(min_tpr, 4),
            'TPR_gap': round(max_tpr - min_tpr, 4)
        })
    pd.DataFrame(eo_summary).to_csv(os.path.join(result_dir, 'equal_opportunity_gap.csv'), index=False)

    # --- demographic_parity_summary.csv ---
    dp_summary = []
    for proxy in df_fair['proxy_feature'].unique():
        sub = df_fair[df_fair['proxy_feature'] == proxy]
        max_pos = sub['Positive Rate (Demographic Parity)'].max()
        min_pos = sub['Positive Rate (Demographic Parity)'].min()
        dp_summary.append({
            'model': model_name,
            'proxy_feature': proxy,
            'max_positive_rate': round(max_pos, 4),
            'min_positive_rate': round(min_pos, 4),
            'dp_gap': round(max_pos - min_pos, 4)
        })
    pd.DataFrame(dp_summary).to_csv(os.path.join(result_dir, 'demographic_parity_summary.csv'), index=False)

    print(f"Fairness evaluation outputs saved to: {result_dir}")


In [50]:
evaluate_fairness_constraints_logreg()


Fairness evaluation outputs saved to: model/fairness_constraints/LogisticRegression_scaled\fairness_result


In [51]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

output_dir = 'model/outputs_compared'
os.makedirs(output_dir, exist_ok=True)

# Strategy mapping per model
strategy_map = {
    'LogisticRegression': {
        'baseline': 'model/original/fairness_result/LogisticRegression',
        'reweighed': 'model/reweighed/fairness_result/LogisticRegression',
        'constraints': 'model/fairness_constraints/LogisticRegression_scaled/fairness_result'
    },
    'RandomForest': {
        'baseline': 'model/original/fairness_result/RandomForest',
        'reweighed': 'model/reweighed/fairness_result/RandomForest'
    },
    'XGBoost': {
        'baseline': 'model/original/fairness_result/XGBoost',
        'reweighed': 'model/reweighed/fairness_result/XGBoost'
    },
    'MLP': {
        'baseline': 'model/original/fairness_result/MLP',
        'reweighed': 'model/reweighed/fairness_result/MLP'
    }
}

# === Metric aggregation ===
def collect_metric(metric_file, value_col, out_csv):
    rows = []
    for model, strategies in strategy_map.items():
        for strategy, path in strategies.items():
            file = os.path.join(path, metric_file)
            if os.path.exists(file):
                df = pd.read_csv(file)
                if 'proxy_feature' in df.columns:
                    for _, r in df.iterrows():
                        rows.append({
                            'model': model,
                            'strategy': strategy,
                            'proxy_feature': r['proxy_feature'],
                            value_col: r[value_col]
                        })
    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(output_dir, out_csv), index=False)
    return df_out

# Collect metrics
df_di = collect_metric('disparate_impact_summary.csv', 'disparate_impact', 'fairness_combined_di.csv')
df_eo = collect_metric('equal_opportunity_gap.csv', 'TPR_gap', 'fairness_combined_eo.csv')
df_dp = collect_metric('demographic_parity_summary.csv', 'dp_gap', 'fairness_combined_dp.csv')

# === Heatmap plotting ===
def plot_heatmap_grouped(df, value_col, out_file, title):
    df['model_strategy'] = df['model'] + '_' + df['strategy']
    pivot = df.pivot(index='proxy_feature', columns='model_strategy', values=value_col)

    row_count = pivot.shape[0]
    col_count = pivot.shape[1]
    fig_height = max(0.6 * row_count, 4)
    fig_width = max(1.2 * col_count, 6)

    plt.figure(figsize=(fig_width, fig_height))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap='coolwarm',
        cbar_kws={'label': value_col},
        annot_kws={"size": 10},
        linewidths=0.3,
        linecolor='gray'
    )
    plt.title(title, fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, out_file), dpi=300)
    plt.close()

# === Bar plot for DP gap ===
def plot_bar(df, value_col, out_file, title):
    df['model_strategy'] = df['model'] + '_' + df['strategy']
    grouped = df.groupby('model_strategy')[value_col].mean().reset_index()
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped, x='model_strategy', y=value_col)
    plt.xticks(rotation=45, ha='right')
    plt.title(title)
    plt.ylabel(value_col)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, out_file))
    plt.close()

# === Final plots ===
plot_heatmap_grouped(df_di, 'disparate_impact', 'heatmap_disparate_impact_combined.png', 'Disparate Impact Comparison')
plot_heatmap_grouped(df_eo, 'TPR_gap', 'heatmap_equal_opportunity_combined.png', 'Equal Opportunity Gap Comparison')
plot_bar(df_dp, 'dp_gap', 'barplot_demographic_parity_combined.png', 'Demographic Parity Gap Comparison')


In [53]:
import os

def list_specified_folders(folder_list, base_indent=''):
    for folder in folder_list:
        print(f"\n🔍 Searching : {folder}")
        list_directory_structure(folder, indent=base_indent)

def list_directory_structure(start_path, indent=''):
    for item in sorted(os.listdir(start_path)):
        path = os.path.join(start_path, item)
        if os.path.isdir(path):
            print(f"{indent}📁 {item}/")
            list_directory_structure(path, indent + '    ')
        else:
            print(f"{indent}📄 {item}")

folders_to_check = [
    'data',
    'model',
    'preprocess',
]

list_specified_folders(folders_to_check)



🔍 Searching : data
📁 GiveMeSomeCredit/
    📄 Data Dictionary.xls
    📄 cs-test.csv
    📄 cs-training.csv
    📄 sampleEntry.csv
📁 archive/
    📄 accepted_2007_to_2018Q4.csv
    📄 rejected_2007_to_2018Q4.csv

🔍 Searching : model
📁 fairness_constraints/
    📁 LogisticRegression_scaled/
        📁 fairness_result/
            📄 demographic_parity_summary.csv
            📄 disparate_impact_summary.csv
            📄 equal_opportunity_gap.csv
            📄 fairness_by_group.csv
        📁 threshold_0.5/
            📄 evaluation_metrics.csv
            📄 test_predictions.csv
        📁 threshold_best/
            📄 evaluation_metrics.csv
            📄 test_predictions.csv
📁 original/
    📁 fairness_result/
        📁 DI/
            📁 LogisticRegression/
                📄 disparate_impact_summary.csv
            📁 MLP/
                📄 disparate_impact_summary.csv
            📁 RandomForest/
                📄 disparate_impact_summary.csv
            📁 XGBoost/
                📄 disparate_impact_